In [1]:
import sqlite3

import pandas as pd
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm


In [2]:
conn = sqlite3.connect("../data/archive.sqlite3")
cursor = conn.cursor()
df = pd.read_sql_query(
    "SELECT * FROM posts,post_tags where posts.thread_id=post_tags.thread_id", conn
)

In [3]:
df

,id,thread_id,user_id,created_at,thanks_count,nothanks_count,raw_html,processed_html,is_first_post,source,thread_id,tag
0,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,Vectors
1,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,geometry
2,2,24681347,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $\Gamma_1$ and $\Gamma_2$ be two circles e...,1,https://artofproblemsolving.com/community/p246...,24681347,geometry
3,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,combinatorics
4,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,geometry
...,...,...,...,...,...,...,...,...,...,...,...,...
1174634,345979,1556697,148231,1.393553e+09,2,0,"<div></div><a href=""http://www.artofproblemsol...",Kyiv Taras Shevchenko University Mechmat Compe...,0,https://artofproblemsolving.com/community/p155...,1556697,inequalities
1174635,345979,1556697,148231,1.393553e+09,2,0,"<div></div><a href=""http://www.artofproblemsol...",Kyiv Taras Shevchenko University Mechmat Compe...,0,https://artofproblemsolving.com/community/p155...,1556697,inequalities unsolved
1174636,345980,1556697,148231,1.398564e+09,2,0,<div></div>The following inequality is also tr...,The following inequality is also true.\nShow t...,0,https://artofproblemsolving.com/community/p155...,1556697,algebra
1174637,345980,1556697,148231,1.398564e+09,2,0,<div></div>The following inequality is also tr...,The following inequality is also true.\nShow t...,0,https://artofproblemsolving.com/community/p155...,1556697,inequalities


In [4]:
df = df[df.is_first_post == 1]
df

,id,thread_id,user_id,created_at,thanks_count,nothanks_count,raw_html,processed_html,is_first_post,source,thread_id,tag
0,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,Vectors
1,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,geometry
2,2,24681347,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $\Gamma_1$ and $\Gamma_2$ be two circles e...,1,https://artofproblemsolving.com/community/p246...,24681347,geometry
3,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,combinatorics
4,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,geometry
...,...,...,...,...,...,...,...,...,...,...,...,...
1174622,345975,1556712,46787,1.247242e+09,2,0,"<div></div>For each nonzero integer <img src=""...",For each nonzero integer $ n$ find all functio...,1,https://artofproblemsolving.com/community/p155...,1556712,algebra unsolved
1174623,345975,1556712,46787,1.247242e+09,2,0,"<div></div>For each nonzero integer <img src=""...",For each nonzero integer $ n$ find all functio...,1,https://artofproblemsolving.com/community/p155...,1556712,function
1174627,345977,1556697,46787,1.247241e+09,1,0,<div></div>Show that for all integers <span st...,"Show that for all integers $ n \ge 2$, $ \sqrt...",1,https://artofproblemsolving.com/community/p155...,1556697,algebra
1174628,345977,1556697,46787,1.247241e+09,1,0,<div></div>Show that for all integers <span st...,"Show that for all integers $ n \ge 2$, $ \sqrt...",1,https://artofproblemsolving.com/community/p155...,1556697,inequalities


In [5]:
df.memory_usage(deep=True)

Index               1346408
id                  1346408
thread_id           1346408
user_id             1346408
created_at          1346408
thanks_count        1346408
nothanks_count      1346408
raw_html          291325751
processed_html     60677999
is_first_post       1346408
source              9861863
thread_id           1346408
tag                 3241616
dtype: int64

In [6]:
tag = df["tag"].value_counts()
tag

tag
geometry           19077
algebra            11493
number theory      11148
combinatorics      10196
inequalities        4823
                   ...  
sets of numbers        1
Prove that             1
airlines               1
cities                 1
pco                    1
Name: count, Length: 3617, dtype: int64

In [7]:
# we 'll take the first 4
tag = tag[tag > 10000]
tag

tag
geometry         19077
algebra          11493
number theory    11148
combinatorics    10196
Name: count, dtype: int64

In [8]:
df = df[df.tag.isin(tag.index)]

In [9]:
df

,id,thread_id,user_id,created_at,thanks_count,nothanks_count,raw_html,processed_html,is_first_post,source,thread_id,tag
1,1,24681373,167643,1.647281e+09,0,0,<div></div>On a board the following six vector...,On a board the following six vectors are writt...,1,https://artofproblemsolving.com/community/p246...,24681373,geometry
2,2,24681347,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $\Gamma_1$ and $\Gamma_2$ be two circles e...,1,https://artofproblemsolving.com/community/p246...,24681347,geometry
3,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,combinatorics
4,3,24681324,167643,1.647280e+09,0,0,"<div></div>Let <img src=""//latex.artofproblems...",Let $P_1P_2...P_n$ be a regular $n$-gon in the...,1,https://artofproblemsolving.com/community/p246...,24681324,geometry
5,4,24681313,167643,1.647280e+09,0,0,"<div></div>Find, with proof, all functions <im...","Find, with proof, all functions $f : R - \{0\}...",1,https://artofproblemsolving.com/community/p246...,24681313,algebra
...,...,...,...,...,...,...,...,...,...,...,...,...
1174607,345970,1556755,46787,1.247244e+09,2,0,"<div></div>Consider a convex solid <img src=""/...",Consider a convex solid $ K$ in space and two ...,1,https://artofproblemsolving.com/community/p155...,1556755,geometry
1174610,345971,1556694,46787,1.247241e+09,1,0,<div></div>Determine the number of integers <i...,Determine the number of integers $ n$ with $ 1...,1,https://artofproblemsolving.com/community/p155...,1556694,number theory
1174615,345973,1556701,46787,1.247241e+09,2,0,<div></div>In a convex quadrilateral <span sty...,"In a convex quadrilateral $ ABCD$, let $ E$ be...",1,https://artofproblemsolving.com/community/p155...,1556701,geometry
1174621,345975,1556712,46787,1.247242e+09,2,0,"<div></div>For each nonzero integer <img src=""...",For each nonzero integer $ n$ find all functio...,1,https://artofproblemsolving.com/community/p155...,1556712,algebra


In [10]:
ds = (
    df.loc[:, ~df.columns.duplicated()]
    .groupby("thread_id", as_index=True)
    .agg(text=("processed_html", "first"), tag=("tag", list))
)
ds

,text,tag
thread_id,,
2,"Let $ABC$ be a triangle, and $M$ an interior p...",[geometry]
3,okay this one is from Prof. Mircea Lascu from ...,"[algebra, geometry]"
5,"If A,B are invertible and the set {Ak - Bk | k...",[algebra]
9,In a magic square $n \times n$ composed from t...,"[algebra, combinatorics]"
72,The lengths of the sides of a convex hexagon $...,[geometry]
...,...,...
36238654,"Let $a, b, c$ be the altitudes of triangle $A$...",[geometry]
36238689,Find all functions that satisfy the condition ...,[algebra]
36238706,On an $N \times N$ “chessboard” ($N \ge 3$) ea...,[combinatorics]


In [11]:
ds.to_csv("out.csv")

In [12]:
ds.memory_usage(deep=True)

Index      371648
text     16118975
tag       3729424
dtype: int64

In [13]:
sp = spm.SentencePieceProcessor(model_file="my_tokenizer.model")

In [14]:
def tokenize(text):
    return sp.encode(text, out_type=int)


X = ds["text"].apply(tokenize)
X

thread_id
2           [215, 3, 427, 7916, 81, 6, 332, 7929, 35, 3, 7...
3           [4808, 232, 149, 275, 29, 264, 387, 7924, 7937...
5           [320, 79, 7929, 7951, 101, 7886, 35, 9, 439, 4...
9           [622, 6, 7885, 471, 3, 7914, 8, 705, 45, 7916,...
72          [266, 2315, 31, 9, 927, 31, 6, 2166, 3058, 3, ...
                                  ...                        
36238654    [215, 3, 7912, 7929, 24, 7929, 18, 7916, 81, 9...
36238689    [1022, 170, 1892, 38, 1023, 9, 733, 98, 170, 5...
36238706    [1782, 22, 3, 7967, 8, 705, 147, 7916, 7909, 0...
36238733    [533, 29, 1689, 38, 156, 1611, 68, 2514, 256, ...
36238754    [622, 332, 3, 427, 48, 3, 311, 138, 217, 7958,...
Name: text, Length: 46456, dtype: object

In [15]:
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(ds["tag"])
print(mlb.classes_)

['algebra' 'combinatorics' 'geometry' 'number theory']


In [16]:
class ContestProblemDataset(Dataset):
    def __init__(self, X, Y):
        self.X = [torch.tensor(x, dtype=torch.long) for x in X]
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


def collate_fn(batch):
    xs, ys = zip(*batch)
    x_lens = torch.tensor([len(x) for x in xs])
    x_padded = pad_sequence(list(xs), batch_first=True, padding_value=8000)
    y_batch = torch.stack(ys)
    return x_padded, y_batch, x_lens

In [17]:
X=X.apply(lambda ids: " ".join(map(str, ids)))

In [18]:
BATCH_SIZE = 16

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=42
)
X_val, X_test, Y_val, Y_test = train_test_split(
    X_test, Y_test, test_size=0.5, random_state=42
)

# train_ds = ContestProblemDataset(X_train, Y_train)
# val_ds = ContestProblemDataset(X_val, Y_val)
# test_ds = ContestProblemDataset(X_test, Y_test)

# train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
# val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
# test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# print(f"Train batches: {len(train_loader)}")
# print(f"Validation batches: {len(val_loader)}")
# print(f"Test batches: {len(test_loader)}")

In [19]:
X

thread_id
2           215 3 427 7916 81 6 332 7929 35 3 7970 7916 22...
3           4808 232 149 275 29 264 387 7924 7937 280 94 9...
5           320 79 7929 7951 101 7886 35 9 439 426 7948 79...
9           622 6 7885 471 3 7914 8 705 45 7916 905 2727 2...
72          266 2315 31 9 927 31 6 2166 3058 3 723 3147 79...
                                  ...                        
36238654    215 3 7912 7929 24 7929 18 7916 81 9 3551 31 3...
36238689    1022 170 1892 38 1023 9 733 98 170 566 362 3 7...
36238706    1782 22 3 7967 8 705 147 7916 7909 0 112 1687 ...
36238733    533 29 1689 38 156 1611 68 2514 256 42 1835 38...
36238754    622 332 3 427 48 3 311 138 217 7958 768 33 629...
Name: text, Length: 46456, dtype: str

In [20]:
X_train

thread_id
21183769    74 25 4440 4482 7929 371 31 846 2355 101 3520 ...
1428739     557 6 1628 439 3 553 7916 31 380 529 7929 356 ...
2165298     242 101 850 3 7914 8 282 1636 4859 31 4678 30 ...
21704410    1022 9 244 31 1734 802 31 9 855 1833 72 7913 7...
7595519     1022 9 1427 863 7943 3363 2184 31 9 244 258 20...
                                  ...                        
3553773     557 331 380 451 3 7950 7916 1624 3 7924 2897 7...
34397733    1782 9 775 3 857 7916 31 1872 3 1340 7916 6 18...
27787094    2152 3 7961 7916 738 95 9 775 3 311 7916 31 33...
337940      215 3 7960 7916 81 9 439 31 170 1734 362 2193 ...
9782029     215 3 1340 7916 81 6 471 7937 266 367 1462 3 3...
Name: text, Length: 32519, dtype: str

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    lowercase=False,                 
    preprocessor=lambda x: x,        
    tokenizer=lambda x: x.split(),   # Tách bằng khoảng trắng
    token_pattern=None               # Tắt regex mặc định
)
X_train = tfidf.fit_transform(X_train)
X_val=tfidf.transform(X_val)
X_test=tfidf.transform(X_test)


In [22]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1807622 stored elements and shape (32519, 7640)>

In [22]:
# Y_train=np.array(Y_train.to_list())

In [64]:
model = xgb.XGBClassifier(
    tree_method="hist", 
    multi_strategy="multi_output_tree",
    objective="binary:logistic", # Vẫn dùng binary vì mỗi cột là 1 xác suất ĐÚNG/SAI độc lập
    n_estimators=200,            # Số cây
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)


In [66]:
model.fit(
    X_train, Y_train,
    eval_set=[(X_train, Y_train), (X_val, Y_val)], # xem cả train và val
    verbose=True # True = in ra mỗi vòng, verbose=10 = in mỗi 10 vòng
)

[0]	validation_0-logloss:0.54124	validation_1-logloss:0.54170
[1]	validation_0-logloss:0.51123	validation_1-logloss:0.51203
[2]	validation_0-logloss:0.48694	validation_1-logloss:0.48782
[3]	validation_0-logloss:0.46516	validation_1-logloss:0.46601
[4]	validation_0-logloss:0.44669	validation_1-logloss:0.44792
[5]	validation_0-logloss:0.43024	validation_1-logloss:0.43192
[6]	validation_0-logloss:0.41348	validation_1-logloss:0.41555
[7]	validation_0-logloss:0.40018	validation_1-logloss:0.40282
[8]	validation_0-logloss:0.38772	validation_1-logloss:0.39045
[9]	validation_0-logloss:0.37689	validation_1-logloss:0.37966
[10]	validation_0-logloss:0.36541	validation_1-logloss:0.36872
[11]	validation_0-logloss:0.35640	validation_1-logloss:0.36012
[12]	validation_0-logloss:0.34811	validation_1-logloss:0.35243
[13]	validation_0-logloss:0.34021	validation_1-logloss:0.34495
[14]	validation_0-logloss:0.33242	validation_1-logloss:0.33749
[15]	validation_0-logloss:0.32580	validation_1-logloss:0.33108
[1

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [ ]:
model2 = xgb.XGBClassifier(
    tree_method="hist", 
    multi_strategy="multi_output_tree",
    objective="binary:logistic", # Vẫn dùng binary vì mỗi cột là 1 xác suất ĐÚNG/SAI độc lập
    n_estimators=200,            # Số cây
    learning_rate=0.1,
    max_depth=6,
    random_state=42,

)
model2.fit(
    X_train, Y_train,
    xgb_model=model.get_booster(),
    eval_set=[(X_train, Y_train), (X_val, Y_val)], # xem cả train và val
    verbose=True # True = in ra mỗi vòng, verbose=10 = in mỗi 10 vòng
)

[0]	validation_0-logloss:0.14903	validation_1-logloss:0.19115
[1]	validation_0-logloss:0.14886	validation_1-logloss:0.19108
[2]	validation_0-logloss:0.14866	validation_1-logloss:0.19103
[3]	validation_0-logloss:0.14842	validation_1-logloss:0.19080
[4]	validation_0-logloss:0.14827	validation_1-logloss:0.19068
[5]	validation_0-logloss:0.14805	validation_1-logloss:0.19063
[6]	validation_0-logloss:0.14781	validation_1-logloss:0.19051
[7]	validation_0-logloss:0.14742	validation_1-logloss:0.19038
[8]	validation_0-logloss:0.14721	validation_1-logloss:0.19034
[9]	validation_0-logloss:0.14696	validation_1-logloss:0.19023
[10]	validation_0-logloss:0.14660	validation_1-logloss:0.19007
[11]	validation_0-logloss:0.14643	validation_1-logloss:0.18999
[12]	validation_0-logloss:0.14620	validation_1-logloss:0.18987
[13]	validation_0-logloss:0.14591	validation_1-logloss:0.18973
[14]	validation_0-logloss:0.14562	validation_1-logloss:0.18960
[15]	validation_0-logloss:0.14542	validation_1-logloss:0.18951
[1

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [76]:
model3 = xgb.XGBClassifier(
    tree_method="hist", 
    multi_strategy="multi_output_tree",
    objective="binary:logistic", # Vẫn dùng binary vì mỗi cột là 1 xác suất ĐÚNG/SAI độc lập
    n_estimators=100,            # Số cây
    learning_rate=0.1,
    max_depth=6,
    random_state=42,

)
model3.fit(
    X_train, Y_train,
    xgb_model=model2.get_booster(),
    eval_set=[(X_train, Y_train), (X_val, Y_val)], # xem cả train và val
    verbose=True # True = in ra mỗi vòng, verbose=10 = in mỗi 10 vòng
)

[0]	validation_0-logloss:0.11515	validation_1-logloss:0.18072
[1]	validation_0-logloss:0.11504	validation_1-logloss:0.18064
[2]	validation_0-logloss:0.11494	validation_1-logloss:0.18064
[3]	validation_0-logloss:0.11480	validation_1-logloss:0.18062
[4]	validation_0-logloss:0.11473	validation_1-logloss:0.18061
[5]	validation_0-logloss:0.11464	validation_1-logloss:0.18058
[6]	validation_0-logloss:0.11456	validation_1-logloss:0.18052
[7]	validation_0-logloss:0.11438	validation_1-logloss:0.18047
[8]	validation_0-logloss:0.11431	validation_1-logloss:0.18047
[9]	validation_0-logloss:0.11420	validation_1-logloss:0.18043
[10]	validation_0-logloss:0.11408	validation_1-logloss:0.18037
[11]	validation_0-logloss:0.11399	validation_1-logloss:0.18035
[12]	validation_0-logloss:0.11388	validation_1-logloss:0.18033
[13]	validation_0-logloss:0.11381	validation_1-logloss:0.18027
[14]	validation_0-logloss:0.11366	validation_1-logloss:0.18025
[15]	validation_0-logloss:0.11341	validation_1-logloss:0.18020
[1

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [79]:
model4 = xgb.XGBClassifier(
    tree_method="hist", 
    multi_strategy="multi_output_tree",
    objective="binary:logistic", # Vẫn dùng binary vì mỗi cột là 1 xác suất ĐÚNG/SAI độc lập
    n_estimators=200,            # Số cây
    learning_rate=0.1,
    max_depth=6,
    random_state=42,

)
model4.fit(
    X_train, Y_train,
    xgb_model=model3.get_booster(),
    eval_set=[(X_train, Y_train), (X_val, Y_val)], # xem cả train và val
    verbose=True # True = in ra mỗi vòng, verbose=10 = in mỗi 10 vòng
)

[0]	validation_0-logloss:0.10463	validation_1-logloss:0.17886
[1]	validation_0-logloss:0.10448	validation_1-logloss:0.17885
[2]	validation_0-logloss:0.10442	validation_1-logloss:0.17884
[3]	validation_0-logloss:0.10428	validation_1-logloss:0.17881
[4]	validation_0-logloss:0.10415	validation_1-logloss:0.17881
[5]	validation_0-logloss:0.10402	validation_1-logloss:0.17882
[6]	validation_0-logloss:0.10385	validation_1-logloss:0.17880
[7]	validation_0-logloss:0.10379	validation_1-logloss:0.17880
[8]	validation_0-logloss:0.10371	validation_1-logloss:0.17877
[9]	validation_0-logloss:0.10362	validation_1-logloss:0.17873
[10]	validation_0-logloss:0.10351	validation_1-logloss:0.17873
[11]	validation_0-logloss:0.10340	validation_1-logloss:0.17870
[12]	validation_0-logloss:0.10329	validation_1-logloss:0.17868
[13]	validation_0-logloss:0.10321	validation_1-logloss:0.17868
[14]	validation_0-logloss:0.10315	validation_1-logloss:0.17864
[15]	validation_0-logloss:0.10308	validation_1-logloss:0.17864
[1

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [82]:
model5 = xgb.XGBClassifier(
    tree_method="hist", 
    multi_strategy="multi_output_tree",
    objective="binary:logistic", # Vẫn dùng binary vì mỗi cột là 1 xác suất ĐÚNG/SAI độc lập
    n_estimators=300,            # Số cây
    learning_rate=0.03,
    max_depth=6,
    random_state=42,

)
model5.fit(
    X_train, Y_train,
    xgb_model=model4.get_booster(),
    eval_set=[(X_train, Y_train), (X_val, Y_val)], # xem cả train và val
    verbose=True # True = in ra mỗi vòng, verbose=10 = in mỗi 10 vòng
)

[0]	validation_0-logloss:0.08827	validation_1-logloss:0.17681
[1]	validation_0-logloss:0.08825	validation_1-logloss:0.17681
[2]	validation_0-logloss:0.08823	validation_1-logloss:0.17679
[3]	validation_0-logloss:0.08821	validation_1-logloss:0.17678
[4]	validation_0-logloss:0.08819	validation_1-logloss:0.17678
[5]	validation_0-logloss:0.08817	validation_1-logloss:0.17677
[6]	validation_0-logloss:0.08815	validation_1-logloss:0.17677
[7]	validation_0-logloss:0.08813	validation_1-logloss:0.17677
[8]	validation_0-logloss:0.08812	validation_1-logloss:0.17677
[9]	validation_0-logloss:0.08810	validation_1-logloss:0.17676
[10]	validation_0-logloss:0.08806	validation_1-logloss:0.17676
[11]	validation_0-logloss:0.08804	validation_1-logloss:0.17675
[12]	validation_0-logloss:0.08802	validation_1-logloss:0.17675
[13]	validation_0-logloss:0.08801	validation_1-logloss:0.17675
[14]	validation_0-logloss:0.08798	validation_1-logloss:0.17675
[15]	validation_0-logloss:0.08797	validation_1-logloss:0.17675
[1

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [29]:
model=xgb.XGBClassifier()
model.load_model("model.json")

In [25]:
model5=xgb.XGBClassifier()
model5.load_model("model5.json")

In [26]:
print(model5.get_booster().num_boosted_rounds()) 

1000


In [143]:
from numpy import average
from sklearn.metrics import jaccard_score, f1_score
proba=model5.predict_proba(X_val)
Y_pred = (proba >= 0.5).astype(int)

# nếu cả 4 đều < 0.5 thì lấy thằng lớn nhất
# all_zero = Y_pred.sum(axis=1) == 0
# Y_pred[all_zero, proba[all_zero].argmax(axis=1)] = 1
# Y_pred=model5.predict(X_test)
print((Y_val == Y_pred).mean(axis=0))
f1_score(Y_val,Y_pred,average="samples")


[0.91661883 0.92896096 0.95752009 0.92192882]


0.8631116396041769

In [144]:
model.save_model("model.json")
model2.save_model("model2.json")
model3.save_model("model3.json")
model4.save_model("model4.json")
model5.save_model("model5.json")

In [57]:
Y_pred=model.predict(X_train)

In [58]:
Y_train

array([[0, 1, 0, 0],
       [1, 0, 0, 0],
       [0, 1, 0, 0],
       ...,
       [0, 0, 1, 0],
       [0, 0, 0, 1],
       [0, 0, 1, 0]], shape=(32519, 4))

In [59]:
Y_pred

array([[0., 1., 0., 0.],
       [0., 0., 0., 1.],
       [0., 1., 0., 0.],
       ...,
       [0., 0., 1., 0.],
       [1., 0., 0., 0.],
       [0., 0., 1., 0.]], shape=(32519, 4))

In [24]:
text = """Let $\\mathbb{N}$ denote the set of positive integers. A function $f\\colon\\mathbb{N}\\to\\mathbb{N}$ is said to be bonza if
\\[
f(a)\\mid b^a-f(b)^{f(a)}
\\]for all positive integers $a$ and $b$.

Determine the smallest real constant $c$ such that $f(n)\\leqslant cn$ for all bonza functions $f$ and all positive integers $n$."""
text2="This is a problem proposed by me"

In [ ]:
model=xgb.XGBClassifier()
model.load_model("model5.json")
inp=tfidf.transform(pd.Series([sp.encode(text)]).apply(lambda ids: " ".join(map(str, ids))))
model.predict_proba(inp)

array([[0.8606497 , 0.01337123, 0.00381078, 0.2882299 ]], dtype=float32)

In [43]:
X

thread_id
2           215 3 427 7916 81 6 332 7929 35 3 7970 7916 22...
3           4808 232 149 275 29 264 387 7924 7937 280 94 9...
5           320 79 7929 7951 101 7886 35 9 439 426 7948 79...
9           622 6 7885 471 3 7914 8 705 45 7916 905 2727 2...
72          266 2315 31 9 927 31 6 2166 3058 3 723 3147 79...
                                  ...                        
36238654    215 3 7912 7929 24 7929 18 7916 81 9 3551 31 3...
36238689    1022 170 1892 38 1023 9 733 98 170 566 362 3 7...
36238706    1782 22 3 7967 8 705 147 7916 7909 0 112 1687 ...
36238733    533 29 1689 38 156 1611 68 2514 256 42 1835 38...
36238754    622 332 3 427 48 3 311 138 217 7958 768 33 629...
Name: text, Length: 46456, dtype: str

In [51]:
Y_test

array([[0, 0, 1, 0],
       [1, 0, 0, 0],
       [0, 0, 1, 0],
       ...,
       [0, 0, 1, 0],
       [0, 1, 0, 0],
       [0, 0, 0, 1]], shape=(6969, 4))

In [55]:
Y_pred=model.predict(X_test)

In [ ]:
import numpy as np
from sklearn.metrics import jaccard_score, f1_score, precision_score, recall_score

def compute_metrics_from_arrays(X_test, Y_test):
    probs = 1 / (1 + np.exp(-X_test))
    preds = (probs >= 0.5).astype(int)
    labels = Y_test.astype(int)

    label_acc = np.mean(preds == labels, axis=0)

    jaccard_samples = jaccard_score(labels, preds, average="samples", zero_division=0)
    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    precision_micro = precision_score(labels, preds, average="micro", zero_division=0)
    recall_micro = recall_score(labels, preds, average="micro", zero_division=0)

    return {
        **{f"acc_label_{i}": float(acc) for i, acc in enumerate(label_acc)},
        "jaccard_samples": float(jaccard_samples),
        "f1_micro": float(f1_micro),
        "f1_macro": float(f1_macro),
        "precision_micro": float(precision_micro),
        "recall_micro": float(recall_micro),
    }

results = compute_metrics_from_arrays(X_test, Y_test)
print(results)

In [94]:
X1_train, X1_test, Y1_train, Y1_test = train_test_split(
    X, Y, test_size=0.3, random_state=42
)
X1_val, X1_test, Y1_val, Y1_test = train_test_split(
    X1_test, Y1_test, test_size=0.5, random_state=42
)

In [95]:
Y1_test

array([[0, 0, 1, 0],
       [1, 0, 0, 0],
       [0, 0, 1, 0],
       ...,
       [0, 0, 1, 0],
       [0, 1, 0, 0],
       [0, 0, 0, 1]], shape=(6969, 4))

In [99]:
df_test=pd.concat([pd.DataFrame(model.predict_proba(X_test)),pd.DataFrame(Y1_test,columns=[4,5,6,7])],axis=1)
df_test.index=X1_test.index

In [100]:
df_test

,0,1,2,3,4,5,6,7
thread_id,,,,,,,,
24870212,0.000214,0.000138,0.999941,0.000286,0,0,1,0
30027564,0.679235,0.038116,0.023292,0.311279,1,0,0,0
24036068,0.000555,0.000291,0.999845,0.000207,0,0,1,0
268382,0.995820,0.000852,0.008129,0.038256,1,0,0,0
17129840,0.010099,0.924488,0.533873,0.004257,0,1,0,0
...,...,...,...,...,...,...,...,...
20972908,0.004340,0.000910,0.998420,0.000737,0,0,1,0
32725963,0.994263,0.003344,0.004340,0.010674,1,0,0,0
31808676,0.043412,0.086676,0.851388,0.087543,0,0,1,0


In [105]:
df_test[(df_test[2]<0.5)&(df_test[6]>0.5)&(df_test[5]<0.5)]

,0,1,2,3,4,5,6,7
thread_id,,,,,,,,
27322324,0.178750,0.498954,0.220463,0.050541,0,0,1,0
14232733,0.381589,0.308786,0.223594,0.067797,0,0,1,0
915985,0.981127,0.007104,0.051303,0.005583,1,0,1,0
1798808,0.850140,0.013793,0.047788,0.118162,0,0,1,0
1986739,0.007218,0.666553,0.250520,0.128052,0,0,1,1
...,...,...,...,...,...,...,...,...
26566078,0.065465,0.444110,0.397772,0.064235,0,0,1,0
34025490,0.177655,0.145327,0.305269,0.292270,0,0,1,0
26751815,0.368617,0.318282,0.420387,0.090187,0,0,1,0


In [92]:
pd.DataFrame(Y1_test)

,0,1,2,3
0,0,0,1,0
1,1,0,0,0
2,0,0,1,0
3,1,0,0,0
4,0,1,0,0
...,...,...,...,...
6964,0,0,1,0
6965,1,0,0,0
6966,0,0,1,0
6967,0,1,0,0


In [90]:
df_test=pd.concat([df_test,pd.DataFrame(Y1_test)])

In [91]:
df_test

,0,1,2,3
24870212,0.000214,0.000138,0.999941,0.000286
30027564,0.679235,0.038116,0.023292,0.311279
24036068,0.000555,0.000291,0.999845,0.000207
268382,0.995820,0.000852,0.008129,0.038256
17129840,0.010099,0.924488,0.533873,0.004257
...,...,...,...,...
6964,0.000000,0.000000,1.000000,0.000000
6965,1.000000,0.000000,0.000000,0.000000
6966,0.000000,0.000000,1.000000,0.000000
6967,0.000000,1.000000,0.000000,0.000000


In [82]:
X1_test.index[fn_indices].to_numpy()

array([ 3607075,  9260243, 25272386, 27322324, 14232733,   915985,
        1798808,  1986739, 17248374,   320036,  2410835,  1358647,
       34748622, 22074856, 10867453,  1034894, 26998986,  2624552,
       20689778,  3462682,  1393141, 22213912, 10949296,  3305723,
        6696361, 32694068, 36058879, 32996086, 26575038, 17150556,
       13252349,   506193, 10549143, 30306766,  2386963,   487376,
        1389512,  2650922,  1567158,  2644130,  2939865, 35324642,
       11100972,  2677579,  1341978,   456566,  2422840, 22152974,
        1124777, 19464937, 33582037, 14970566,  1999385,   466207,
       27720516, 31293779, 34074602,  3492744, 24681373,  5331164,
        5852495,  3269345, 12602514,  3041827,  3490345, 21040339,
        3584572,  2681123, 34104714,  3515251,  3425592, 19751495,
       21748540,   849671,  2260935, 22161304, 18349960,   337820,
         364402,  3076921,   849665,  1115387, 17195562,  1327713,
        2594748,   860103, 32810158, 23959314, 11119206,  1604

In [83]:
c=np.column_stack([X1_test.index[fn_indices].to_numpy(),model.predict_proba(X_test)[fn_indices]])